# CORE-trm: Evaluation & Analysis

Comprehensive evaluation including:
- Overall metrics (Recall@20, MRR@20)
- Grouped metrics by prefix length
- Grouped metrics by target item popularity
- Comparison with CORE-ave and baselines
- Attention weight analysis
- Qualitative analysis of predictions

In [ ]:
import json
import torch
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path
import sys
sys.path.append("../../src")

from models.core_trm import CoreTrm, create_core_trm
from models.core_ave import CoreAve, create_core_ave
from data.datasets import create_dataloaders, SessionDataset
from evaluation.metrics import (
    evaluate_model, recall_at_k, mrr_at_k,
    group_by_prefix_length, group_by_target_popularity,
    PopularityRecommender, TransitionRecommender, SessionPopularityRecommender
)

In [ ]:
# Load Data & Vocabulary
train_df = pd.read_parquet("../../data/processed/train.parquet")
val_df = pd.read_parquet("../../data/processed/validation.parquet")
test_df = pd.read_parquet("../../data/processed/test.parquet")

with open("../../data/processed/item_mapping.json") as f:
    mapping = json.load(f)
item2idx = mapping['item2idx']
idx2item = {int(k): v for k, v in mapping['idx2item'].items()}
vocab_size = mapping['vocab_size']

print(f"Train: {len(train_df):,} pairs")
print(f"Val: {len(val_df):,} pairs")
print(f"Test: {len(test_df):,} pairs")
print(f"Vocab size: {vocab_size}")

In [ ]:
# Build Train Frequency & Transitions
train_items = []
for row in train_df.itertuples():
    train_items.extend(row.prefix_items)
    train_items.append(row.target_item)
train_freq = Counter(train_items)

transitions = {}
for row in train_df.itertuples():
    if len(row.prefix_items) > 0:
        last = row.prefix_items[-1]
        nxt = row.target_item
        if last not in transitions:
            transitions[last] = Counter()
        transitions[last][nxt] += 1

print(f"Train vocab size: {len(train_freq)}")
print(f"Transition pairs: {len(transitions)}")

In [ ]:
# Load Both Models
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# CORE-trm
model_trm = create_core_trm(vocab_size=vocab_size).to(device)
ckpt_trm = torch.load("../../checkpoints/core_trm/core_trm_seed42_best.pt", map_location=device)
model_trm.load_state_dict(ckpt_trm["model_state_dict"])
model_trm.eval()

# CORE-ave
model_ave = create_core_ave(vocab_size=vocab_size).to(device)
ckpt_ave = torch.load("../../checkpoints/core_ave/core_ave_seed42_best.pt", map_location=device)
model_ave.load_state_dict(ckpt_ave["model_state_dict"])
model_ave.eval()

print(f"CORE-trm loaded from epoch {ckpt_trm['epoch']}")
print(f"CORE-ave loaded from epoch {ckpt_ave['epoch']}")

In [ ]:
# Evaluate Both Models
print("Evaluating CORE-trm on test set...")
trm_results = evaluate_model(
    model_trm, test_df, k=20,
    item2idx=item2idx, idx2item=idx2item,
    verbose=True
)
trm_results['by_target_popularity'] = group_by_target_popularity(
    trm_results['per_example'], train_freq
)

print("\nEvaluating CORE-ave on test set...")
ave_results = evaluate_model(
    model_ave, test_df, k=20,
    item2idx=item2idx, idx2item=idx2item,
    verbose=True
)
ave_results['by_target_popularity'] = group_by_target_popularity(
    ave_results['per_example'], train_freq
)

In [ ]:
# Evaluate Baselines
print("\nEvaluating baselines...")

pop_model = PopularityRecommender(train_freq, k=20)
pop_results = evaluate_model(pop_model, test_df, k=20, verbose=True)
pop_results['by_target_popularity'] = group_by_target_popularity(pop_results['per_example'], train_freq)

trans_model = TransitionRecommender(transitions, k=20)
trans_results = evaluate_model(trans_model, test_df, k=20, verbose=True)
trans_results['by_target_popularity'] = group_by_target_popularity(trans_results['per_example'], train_freq)

sesspop_model = SessionPopularityRecommender(train_freq, k=20)
sesspop_results = evaluate_model(sesspop_model, test_df, k=20, verbose=True)
sesspop_results['by_target_popularity'] = group_by_target_popularity(sesspop_results['per_example'], train_freq)

In [ ]:
# Overall Comparison Table
print("\n" + "="*70)
print("OVERALL METRICS COMPARISON")
print("="*70)
print(f"{'Model':<20} | {'Recall@20':>10} | {'MRR@20':>10}")
print("-"*70)

models = [
    ("CORE-trm", trm_results),
    ("CORE-ave", ave_results),
    ("Transition", trans_results),
    ("Session-POP", sesspop_results),
    ("POP", pop_results),
]

for name, res in models:
    print(f"{name:<20} | {res['overall']['recall@20']:>10.4f} | {res['overall']['mrr@20']:>10.4f}")

In [ ]:
# By Prefix Length Comparison
print("\n" + "="*70)
print("BY PREFIX LENGTH")
print("="*70)

bins = ['1', '2-3', '4-6', '7+']

print(f"{'Model':<15} | {'1':^15} | {'2-3':^15} | {'4-6':^15} | {'7+':^15}")
print("-"*80)

for name, res in models:
    row = f"{name:<15}"
    for b in bins:
        m = res['by_prefix_length'].get(b, {'recall@20': 0, 'mrr@20': 0, 'count': 0})
        row += f" | R={m['recall@20']:.3f} M={m['mrr@20']:.3f} n={m['count']}"
    print(row)

In [ ]:
# By Target Popularity Comparison
print("\n" + "="*70)
print("BY TARGET POPULARITY (Train Frequency Tertiles)")
print("="*70)

pop_bins = ['head', 'torso', 'tail']

print(f"{'Model':<15} | {'Head':^20} | {'Torso':^20} | {'Tail':^20}")
print("-"*80)

for name, res in models:
    row = f"{name:<15}"
    for b in pop_bins:
        m = res['by_target_popularity'].get(b, {'recall@20': 0, 'mrr@20': 0, 'count': 0, 'freq_range': (0,0)})
        row += f" | R={m['recall@20']:.3f} M={m['mrr@20']:.3f} n={m['count']}"
    print(row)

In [ ]:
# Attention Weight Analysis (CORE-trm only)
print("\n" + "="*70)
print("ATTENTION WEIGHT ANALYSIS (CORE-trm)")
print("="*70)

test_loader = create_dataloaders(
    train_path="../../data/processed/train.parquet",
    val_path="../../data/processed/validation.parquet",
    test_path="../../data/processed/test.parquet",
    item_mapping_path="../../data/processed/item_mapping.json",
    batch_size=32,
    max_len=50,
    num_workers=0
)[2]

model_trm.eval()
alpha_stats = []

with torch.no_grad():
    for prefix, target, plen in test_loader:
        prefix = prefix.to(device)
        plen = plen.to(device)
        
        _, _, alpha = model_trm(prefix, plen)
        
        for i in range(prefix.size(0)):
            valid_len = plen[i].item()
            valid_alpha = alpha[i, -valid_len:].cpu().numpy()
            
            alpha_stats.append({
                'valid_len': valid_len,
                'max_weight': valid_alpha.max(),
                'min_weight': valid_alpha.min(),
                'mean_weight': valid_alpha.mean(),
                'std_weight': valid_alpha.std(),
                'entropy': -np.sum(valid_alpha * np.log(valid_alpha + 1e-10)),
                'max_pos': np.argmax(valid_alpha),
            })
        
        if len(alpha_stats) >= 1000:
            break

alpha_df = pd.DataFrame(alpha_stats)
print(f"Analyzed {len(alpha_df)} samples")
print(f"\nAttention weight statistics:")
print(f"  Mean max weight: {alpha_df['max_weight'].mean():.4f}")
print(f"  Mean entropy: {alpha_df['entropy'].mean():.4f}")
print(f"  Mean max position (from end): {alpha_df['max_pos'].mean():.2f}")
print(f"  Position 0 (most recent) is max: {(alpha_df['max_pos'] == 0).mean()*100:.1f}%")
print(f"  Position -1 (second recent) is max: {(alpha_df['max_pos'] == 1).mean()*100:.1f}%")

# By session length
for length in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:
    subset = alpha_df[alpha_df['valid_len'] == length]
    if len(subset) > 0:
        print(f"  Length {length}: entropy={subset['entropy'].mean():.3f}, max_pos={subset['max_pos'].mean():.2f}")

In [ ]:
# Attention Visualization for Sample Sessions
import matplotlib.pyplot as plt
import seaborn as sns

def get_item_name(item_id):
    catalog = {
        81766: "Wireless Headphones Pro",
        31331: "Bluetooth Speaker Mini",
        32118: "USB-C Hub 7-in-1",
        9654: "Mechanical Keyboard RGB",
        5103: "Ergonomic Mouse Vertical",
        69597: "27\" 4K Monitor IPS",
        4168: "Laptop Stand Aluminum",
        79898: "Webcam 1080p Streaming",
        36083: "External SSD 1TB NVMe",
        10858: "WiFi 6 Router AX3000",
        14614: "Smart Desk Lamp LED",
        6848: "Cable Management Kit",
        79141: "Monitor Arm Dual",
        12345: "Gaming Chair Ergonomic",
        34567: "Keycap Set PBT Cherry",
        78901: "RGB Light Strip 2m",
        89012: "Microphone Arm Boom",
        1234: "Acoustic Panels 12pk",
        5678: "Stream Deck Mini",
        9012: "Green Screen Collapsible",
    }
    return catalog.get(item_id, f"Item {item_id}")

# Get a few sample sessions with attention weights
model_trm.eval()
samples = []

with torch.no_grad():
    for prefix, target, plen in test_loader:
        prefix = prefix.to(device)
        plen = plen.to(device)
        
        _, _, alpha = model_trm(prefix, plen)
        
        for i in range(prefix.size(0)):
            valid_len = plen[i].item()
            if valid_len >= 3:
                items = prefix[i, -valid_len:].cpu().tolist()
                target_item = target[i].item()
                weights = alpha[i, -valid_len:].cpu().numpy()
                
                samples.append({
                    'items': items,
                    'target': target_item,
                    'weights': weights,
                    'length': valid_len
                })
                
                if len(samples) >= 6:
                    break
        
        if len(samples) >= 6:
            break

# Plot attention weights
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, sample in enumerate(samples):
    ax = axes[idx]
    items = [get_item_name(x) for x in sample['items']]
    weights = sample['weights']
    
    bars = ax.bar(range(len(items)), weights, color='steelblue', edgecolor='white')
    ax.set_xticks(range(len(items)))
    ax.set_xticklabels(items, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Attention Weight')
    ax.set_title(f"Len={sample['length']}, Target={get_item_name(sample['target'])}", fontsize=10)
    ax.set_ylim(0, max(weights) * 1.2)
    
    # Highlight max weight
    max_idx = np.argmax(weights)
    bars[max_idx].set_color('orange')

plt.tight_layout()
plt.savefig("../../reports/models/core_trm_attention_weights.png", dpi=150, bbox_inches='tight')
plt.close()
print("Attention weight visualization saved")

In [ ]:
# Qualitative Analysis
print("\n" + "="*70)
print("QUALITATIVE ANALYSIS")
print("="*70)

def get_item_name(item_id):
    catalog = {
        81766: "Wireless Headphones Pro",
        31331: "Bluetooth Speaker Mini",
        32118: "USB-C Hub 7-in-1",
        9654: "Mechanical Keyboard RGB",
        5103: "Ergonomic Mouse Vertical",
        69597: "27\" 4K Monitor IPS",
        4168: "Laptop Stand Aluminum",
        79898: "Webcam 1080p Streaming",
        36083: "External SSD 1TB NVMe",
        10858: "WiFi 6 Router AX3000",
        14614: "Smart Desk Lamp LED",
        6848: "Cable Management Kit",
        79141: "Monitor Arm Dual",
        12345: "Gaming Chair Ergonomic",
        34567: "Keycap Set PBT Cherry",
        78901: "RGB Light Strip 2m",
        89012: "Microphone Arm Boom",
        1234: "Acoustic Panels 12pk",
        5678: "Stream Deck Mini",
        9012: "Green Screen Collapsible",
    }
    return catalog.get(item_id, f"Item {item_id}")

# Compare CORE-trm vs CORE-ave predictions
print("--- Sample where CORE-trm differs from CORE-ave ---")

divergent = []
with torch.no_grad():
    for prefix, target, plen in test_loader:
        prefix = prefix.to(device)
        plen = plen.to(device)
        
        trm_scores, _, _ = model_trm(prefix, plen)
        ave_scores, _ = model_ave(prefix, plen)
        
        trm_top5 = torch.topk(trm_scores, k=5, dim=1).indices
        ave_top5 = torch.topk(ave_scores, k=5, dim=1).indices
        
        for i in range(prefix.size(0)):
            trm_recs = trm_top5[i].cpu().tolist()
            ave_recs = ave_top5[i].cpu().tolist()
            target_item = target[i].item()
            
            trm_rank = trm_recs.index(target_item) + 1 if target_item in trm_recs else -1
            ave_rank = ave_recs.index(target_item) + 1 if target_item in ave_recs else -1
            
            if trm_rank != ave_rank:
                divergent.append({
                    'prefix': prefix[i, -plen[i]:].cpu().tolist(),
                    'target': target_item,
                    'trm_recs': trm_recs,
                    'ave_recs': ave_recs,
                    'trm_rank': trm_rank,
                    'ave_rank': ave_rank
                })
                
                if len(divergent) >= 5:
                    break
        
        if len(divergent) >= 5:
            break

for ex in divergent:
    session = [get_item_name(x) for x in ex['prefix'][-3:]]
    target = get_item_name(ex['target'])
    trm_recs = [get_item_name(x) for x in ex['trm_recs'][:5]]
    ave_recs = [get_item_name(x) for x in ex['ave_recs'][:5]]
    
    print(f"  Session: {' -> '.join(session)} -> Target: {target}")
    print(f"    CORE-trm: {trm_recs} (rank: {ex['trm_rank']})")
    print(f"    CORE-ave: {ave_recs} (rank: {ex['ave_rank']})")
    print()

In [ ]:
# Save Evaluation Results
eval_output = {
    "model": "core_trm",
    "overall": trm_results["overall"],
    "by_prefix_length": trm_results["by_prefix_length"],
    "by_target_popularity": trm_results["by_target_popularity"],
    "comparison_with_core_ave": {
        "core_ave_overall": ave_results["overall"],
        "core_trm_overall": trm_results["overall"]
    },
    "baselines": {
        "pop": {"overall": pop_results["overall"]},
        "transition": {"overall": trans_results["overall"]},
        "session_pop": {"overall": sesspop_results["overall"]}
    },
    "attention_analysis": {
        "mean_max_weight": float(alpha_df['max_weight'].mean()),
        "mean_entropy": float(alpha_df['entropy'].mean()),
        "mean_max_position": float(alpha_df['max_pos'].mean()),
        "recent_is_max_pct": float((alpha_df['max_pos'] == 0).mean() * 100)
    }
}

with open("../../reports/models/core_trm_evaluation.json", 'w') as f:
    json.dump(eval_output, f, indent=2, default=str)

print("\nEvaluation results saved to ../../reports/models/core_trm_evaluation.json")